In [ ]:
    
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime 
import sys
import warnings

FILE_NAME = 'Nat_Gas.csv'

class Utilities():

    @staticmethod
    def process_x(x_val) -> int:
        native = int(x_val.timestamp() * 10**9)/1000000000000000
        return native

    # @staticmethod
    # def deprocess_x(x_val: int) -> TimeStamp:
    #     x = (x_val / 10 ** 9) * 1000000000000000
    #     deprocess = datetime.datetime.utcfromtimestamp(x)
    #     return deprocess

    @staticmethod
    def process_x_from_float(x_val: float) -> int:
        native = int(x_val * 10**9)/1000000000000000
        return native

    @staticmethod
    def interp_y_from_x(input : int, x : float, y : float) -> float:
        return np.interp(input, x, y)

    @staticmethod
    def calc_y_fit(x : float,y : float):
        # Add some noise to the data
        y_noise = y + np.random.normal(0, 0.03, x.shape)
        
        # Fit a sine curve to the noisy data with a higher degree polynomial
        p = np.polyfit(x, y_noise, 50)
        y_fit = np.polyval(p, x)
    
        return y_fit

class PredictPrice():

    def __init__(self, dateIn : str):
        self.dateIn = dateIn
        self.fileName = FILE_NAME

    def train_prep_model(self, data):
        # Load the synthetic data
        
        series = data['Prices']
        
        # Fit SARIMA model
        model = SARIMAX(series, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
        model_fit = model.fit()
        
        # Forecast future values
        forecast = model_fit.forecast(steps=12)
    
        series_new = data['Prices']

        df = pd.DataFrame({'1':series_new,'2':forecast})
    
        # Combine into two lists
        x = []
        y = []
        for a,b in series_new.items():
            # print("process " + str(a))
            native = Utilities.process_x(a)
            x.append(native)
            y.append(b)        
        
        x = np.array(x)
        y = np.array(y)
        return x,y
    
    def runner(self) -> float:
        data = pd.read_csv(self.fileName, index_col='Dates', parse_dates=True)
        x,y = self.train_prep_model(data)
        
        
        y_fit = Utilities.calc_y_fit(x,y)
        # plot_me(x,y, y_fit)
        y = np.array(y_fit)
        
        # y should be sorted for both of these methods
        order = y.argsort()
        y = y[order]
        x = x[order]
    
        prediction = 0.0
        # try:
        dt = datetime.datetime.strptime(self.dateIn, "%m/%d/%Y")
        res = dt.timestamp()
        res = Utilities.process_x_from_float(res)
        
        prediction = Utilities.interp_y_from_x(res, x, y)
        # except:
            
        #     prediction = 0.0
    
        return float(prediction)

In [ ]:
import datetime
class PrototypePricing:
    def __init__(self, units: str, transactionDates: list,
                 injectWithdrawalRate: tuple, maxVolume: float, monthlyStorage: float,
                 transportCost: float):
        
        self.units = units
        self.transactionDates = transactionDates
        self.injectWithdrawalRate = injectWithdrawalRate
        self.maxVolume = maxVolume
        self.monthlyStorage = monthlyStorage
        self.transportCost = transportCost

        # calculated variables
        self.transactionCount = 0.0

        # organizing transactions by date
        self.sortTransactions()
        # self.toString() 

    def toString(self):
        print("PrototypePricing")
        print("units:"+str(self.units))
        print("transactionDates:"+str(self.transactionDates))
        print("injectWithdrawalRate:"+str(self.injectWithdrawalRate))
        print("maxVolume:"+str(self.maxVolume))
        print("monthlyStorage:"+str(self.monthlyStorage))
        print("transportCost:"+str(self.transportCost))

    def sortTransactions(self):
        print("Before: " + str(self.transactionDates))
        self.transactionDates.sort(key=lambda x: datetime.datetime.strptime(x[0], '%m/%d/%Y'))
        print("After: " + str(self.transactionDates))

    
    def calculateValue(self) -> float:

        value = 0.0
        
        if len(self.transactionDates) == 0:
            return 0.0
        
        quantitySum = 0.0
        for i in range(len(self.transactionDates)):
            quantity = self.transactionDates[i][1]
            onDate = self.transactionDates[i][0]

            # Add up injection amount
            quantitySum += quantity
            self.transactionCount += 1

            if quantitySum > self.maxVolume:
                print("max volume exceeded")
                break

            if quantitySum < 0:
                print("negative quantity exceeded on date " + onDate)
                break

            pp = PredictPrice(onDate)
            unitPrice = pp.runner()
            print("quanity on " + onDate + " is " + str(quantity) + " and unit price is " + str(unitPrice))
            value += (quantity * unitPrice)

            # apply inject/withdrawal rate
            priceRate = self.injectWithdrawalRate[0]
            quantityRate = self.injectWithdrawalRate[1]
            injectWithdrawalCost = (priceRate/quantityRate) * abs(quantity)
            value -= injectWithdrawalCost
            print("inject/withdrawal cost is " + str(injectWithdrawalCost))


        # Charge transport cost for each transaction
        fullTransactionCost = self.transportCost * self.transactionCount
        
        print("full transaction cost is " + str(fullTransactionCost))
        value -= fullTransactionCost

        return value

        

In [ ]:
units = "MMBtu"
transactionDates = [("6/1/2023", 1000000),("10/1/2023", -1000000)]
injectionWithdrawalRate = (10000, 1000000)
maxVolume = 2000000
monthlyStorage = 100000                      # monthly rate
transportCost = 50000

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

pp = PrototypePricing(units, transactionDates, 
                     injectionWithdrawalRate, maxVolume, monthlyStorage,
                     transportCost)

print("Total value: " + str(pp.calculateValue()))


Before: [('6/1/2023', 1000000), ('10/1/2023', -1000000)]
After: [('6/1/2023', 1000000), ('10/1/2023', -1000000)]
quanity on 6/1/2023 is 1000000 and unit price is 11.486857158550318
inject/withdrawal cost is 5000.0
quanity on 10/1/2023 is -1000000 and unit price is 11.571045614125437
inject/withdrawal cost is 5000.0
full transaction cost is 100000.0
Total value: -194188.45557511784


In [ ]:
from task1 import PredictPrice
import warnings
import sys

if __name__ == "__main__": 
    # Check if the correct number of arguments is provided
    warnings.filterwarnings('ignore', category=UserWarning)
    warnings.filterwarnings('ignore', category=RuntimeWarning)
    
    if len(sys.argv) < 7:
        print("Usage example: python script.py 'MMBtu' '[('6/1/2023', 1000000),('10/1/2023', -1000000)]' '(10000, 1000000)' 2000000 100000 50000")
        exit(0)


    # Accessing arguments
    script_name = sys.argv[0]
    first_argument = sys.argv[1]
    second_argument = sys.argv[2]
    third_argument = sys.argv[3]
    fourth_argument = sys.argv[4]
    fifth_argument = sys.argv[5]
    sixth_argument = sys.argv[6]

    units = first_argument
    transactionDates = eval(second_argument)
    injectionWithdrawalRate = eval(third_argument)
    maxVolume = float(fourth_argument)
    monthlyStorage = float(fifth_argument)
    transportCost = float(sixth_argument)

    # units = "MMBtu"
    # transactionDates = [("6/1/2023", 1000000),("10/1/2023", -1000000)]
    # injectionWithdrawalRate = (10000, 1000000)
    # maxVolume = 2000000
    # monthlyStorage = 100000                      # monthly rate
    # transportCost = 50000

    warnings.filterwarnings('ignore', category=UserWarning)
    warnings.filterwarnings('ignore', category=RuntimeWarning)

    pp = PrototypePricing(units, transactionDates, 
                        injectionWithdrawalRate, maxVolume, monthlyStorage,
                        transportCost)

    print("Total value: " + str(pp.calculateValue()))
    
    
